# Environment Requirements
To run this repository successfully, the following environment is required:
- **GPU**: NVIDIA A100 with 80GB High RAM
- **Python**: 3.10 or higher
- **CUDA**: 12.1 or higher
- **PyTorch**: 2.1 or higher

In [3]:
import sys
import os
import subprocess
import multiprocessing
import gc
import torch
from huggingface_hub import login

In [9]:
# --- HUGGING FACE TOKEN INSTRUCTIONS ---
# 1. Go to https://huggingface.co/settings/tokens
# 2. Click on "New token" to create a token (Read access is usually sufficient).
# 3. Copy the token and replace the placeholder string below.
# ---------------------------------------

HF_TOKEN = "hf_xxx_your_token_here"

In [10]:
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
!hf auth login --token $HF_TOKEN

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `HF_TOKEN` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
# Set the start method globally once
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

def cleanup_vllm_and_gpu():
    """
    Resets the vLLM singleton and clears GPU memory cache.
    Call this after every benchmark run to ensure stability.
    """
    # Reset the LLM singleton if it exists
    try:
        import utils.llm as _llm_mod
        if _llm_mod._backend_instance is not None:
            print("[Cleanup] Resetting stale LLM singleton …")
            _llm_mod._backend_instance = None
    except Exception as e:
        print(f"[Cleanup] LLM reset skipped or failed: {e}")

    # Clear GPU memory
    try:
        gc.collect()
        torch.cuda.empty_cache()
        free_mem = torch.cuda.mem_get_info()[0] / 1e9
        print(f"[Cleanup] GPU memory freed. Available: {free_mem:.1f} GB")
    except Exception as e:
        print(f"[Cleanup] Memory clearing failed: {e}")

    print("[Cleanup] Ready for next run.")

def setup_git_repo(repo_url, base_path="/content"):
    """
    Clones a git repository or pulls updates if it already exists.

    Args:
        repo_url (str): The HTTPS URL of the GitHub repository.
        base_path (str): The local directory where the repo should live.
    """
    # Ensure we are in the correct base directory
    os.chdir(base_path)

    # Extract the repository name from the URL (e.g., 'raise-26')
    repo_dir = repo_url.split("/")[-1].replace(".git", "")
    repo_path = os.path.join(base_path, repo_dir)

    if os.path.exists(repo_path):
        print(f"Directory '{repo_dir}' exists. Pulling latest changes...")
        # Run git pull inside the specific directory
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    else:
        print(f"Directory '{repo_dir}' not found. Cloning...")
        subprocess.run(["git", "clone", repo_url], check=True)

    return repo_path

def check_environment():
    # Check Hugging Face Token
    assert HF_TOKEN != "hf_xxx_your_token_here" and HF_TOKEN.startswith("hf_"), (
        "Please replace the placeholder with a valid Hugging Face token starting with 'hf_'."
    )

    # Check Python version
    assert sys.version_info >= (3, 10), f"Python version must be >= 3.10. Found {sys.version.split()[0]}"

    # Check PyTorch version
    torch_version = torch.__version__
    assert int(torch_version.split('.')[0]) >= 2, f"PyTorch version must be >= 2.0. Found {torch_version}"

    # Check CUDA availability
    assert torch.cuda.is_available(), "CUDA is not available. Please enable a GPU runtime."

    # Check CUDA version
    cuda_version = torch.version.cuda
    assert cuda_version is not None and float('.'.join(cuda_version.split('.')[:2])) >= 12.1, f"CUDA version must be >= 12.1. Found {cuda_version}"

    # Check GPU type and memory
    device_name = torch.cuda.get_device_name(0)
    assert "A100" in device_name, f"An A100 GPU is required. Found: {device_name}"

    total_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    assert total_memory_gb >= 79, f"GPU memory must be at least 80GB. Found: {total_memory_gb:.1f}GB"

    print("All environment checks passed successfully! ✅")

In [11]:
check_environment()

All environment checks passed successfully! ✅


In [9]:
%%capture
%pip install -U vllm --extra-index-url https://wheels.vllm.ai/nightly
%pip install chromadb flask-cors flask-socketio "git+https://github.com/huggingface/transformers.git"
%pip install -U huggingface_hub
%pip install "https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.9.0/flash_attn-2.8.3+cu128torch2.10-cp312-cp312-linux_x86_64.whl"

In [10]:
# Patch opentelemetry version conflict: vllm nightly ships SDK 1.41.0
# whose _logs exporter expects _ON_EMIT_RECURSION_COUNT_KEY in the API,
# but the matching API build is missing it.

# 1. Moneky Patch to define it because it's absent
# import opentelemetry.context as _otel_ctx
# if not hasattr(_otel_ctx, '_ON_EMIT_RECURSION_COUNT_KEY'):
#     _otel_ctx._ON_EMIT_RECURSION_COUNT_KEY = _otel_ctx.create_key('on_emit_recursion_count')
#     print("[Patch] Injected missing _ON_EMIT_RECURSION_COUNT_KEY into opentelemetry.context")
# else:
#     print("[Patch] _ON_EMIT_RECURSION_COUNT_KEY already present — no fix needed")

# 2. Disabling telemetry
os.environ["OTEL_SDK_DISABLED"] = "true"

In [7]:
%cd /content
setup_git_repo("https://github.com/pandevim/TriMem.git")
%cd /content/TriMem

/content
Directory 'TriMem' exists. Pulling latest changes...
/content/TriMem


In [ ]:
%run run_benchmark.py --agent baseline --tasks 1

In [ ]:
%run run_benchmark.py --agent visual_bus --tasks 1

In [ ]:
%run run_benchmark.py --agent rag --tasks 1

In [ ]:
%run run_benchmark.py --agent msa --tasks 1

In [ ]:
%run run_benchmark.py --agent visual_bus_rag --tasks 1